#### CAMADA 3 — XAI (Explainable AI)
#### Classificacao tematica via LDiA + Relatorio explicavel por PADO
#### Combina resultados do notebook 1 (conformidade R1-R7) e
#### notebook 2 (conformance checking 4 perspectivas)

In [35]:
import pickle
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

print("Imports carregados")

Imports carregados


In [36]:
# 2. CONFIGURACAO
PASTA_PADOS    = r"D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel"
ARQUIVO_NB1    = r"resultado_requisitos_pados_sf.xlsx"
ARQUIVO_NB2    = r"event_log_instauracao.csv"
ARQUIVO_LDA    = r"modelo_lda_pados.pkl"
ARQUIVO_SAIDA  = r"relatorio_xai_pados.xlsx"

print(f"Pasta PADOs : {Path(PASTA_PADOS).resolve()}")
print(f"Notebook 1  : {Path(ARQUIVO_NB1).resolve()}")
print(f"Notebook 2  : {Path(ARQUIVO_NB2).resolve()}")
print(f"Modelo LDiA : {Path(ARQUIVO_LDA).resolve()}")
print(f"Saida XAI   : {Path(ARQUIVO_SAIDA).resolve()}")

Pasta PADOs : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel
Notebook 1  : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\resultado_requisitos_pados_sf.xlsx
Notebook 2  : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\event_log_instauracao.csv
Modelo LDiA : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\modelo_lda_pados.pkl
Saida XAI   : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\relatorio_xai_pados.xlsx


In [37]:
# 3. CARREGAR RESULTADOS DOS NOTEBOOKS ANTERIORES

# Notebook 1 — conformidade R1-R7 com explicacoes
df_nb1 = pd.read_excel(ARQUIVO_NB1, sheet_name='Detalhes e Evidências')
print(f"Notebook 1 carregado: {len(df_nb1)} PADOs")
print(f"Colunas: {list(df_nb1.columns)}")
print()

# Notebook 2 — event log com atividades e timestamps
df_nb2 = pd.read_csv(ARQUIVO_NB2, sep=';', encoding='utf-8-sig')
df_nb2['timestamp'] = pd.to_datetime(df_nb2['timestamp'], errors='coerce')
print(f"Notebook 2 carregado: {len(df_nb2)} eventos")
print(f"PADOs no event log: {df_nb2['case_id'].nunique()}")
print()

# Agrupa atividades por PADO para uso posterior
atividades_por_pado = defaultdict(set)
for _, row in df_nb2.iterrows():
    atividades_por_pado[row['case_id']].add(row['activity'])

print(f"Atividades mapeadas para {len(atividades_por_pado)} PADOs")

Notebook 1 carregado: 104 PADOs
Colunas: ['PADO', 'Processo', 'Autuado', 'Tipo_PADO', 'Qtd_PDFs', 'Tipos_Documentos', 'R1_Documento_Motivador', 'R1_Documento_Motivador_evidencia', 'R1_Documento_Motivador_etapa', 'R1_Documento_Motivador_explicacao', 'R2_Analise_Previa', 'R2_Analise_Previa_evidencia', 'R2_Analise_Previa_etapa', 'R2_Analise_Previa_explicacao', 'R3_Despacho_Instauracao', 'R3_Despacho_Instauracao_evidencia', 'R3_Despacho_Instauracao_etapa', 'R3_Despacho_Instauracao_explicacao', 'R4_Notificacao_Autuado', 'R4_Notificacao_Autuado_evidencia', 'R4_Notificacao_Autuado_etapa', 'R4_Notificacao_Autuado_explicacao', 'R5_Base_Legal', 'R5_Base_Legal_evidencia', 'R5_Base_Legal_etapa', 'R5_Base_Legal_explicacao', 'R6_Prazo_Defesa', 'R6_Prazo_Defesa_evidencia', 'R6_Prazo_Defesa_etapa', 'R6_Prazo_Defesa_explicacao', 'R7_Identificacao_Processo', 'R7_Identificacao_Processo_evidencia', 'R7_Identificacao_Processo_etapa', 'R7_Identificacao_Processo_explicacao', 'Conformidade', 'Pct', 'paginas_s

In [38]:
# 4. CARREGAR MODELO LDiA DO NOTEBOOK 1b

with open(ARQUIVO_LDA, "rb") as f:
    artefatos_lda = pickle.load(f)

lda_modelo    = artefatos_lda["lda_modelo"]
vetorizador   = artefatos_lda["vetorizador"]
MAPA_TOPICOS  = artefatos_lda["MAPA_TOPICOS"]
N_TOPICOS_LDA = artefatos_lda["N_TOPICOS_LDA"]

print(f"Modelo LDiA carregado")
print(f"Topicos: {N_TOPICOS_LDA}")
print(f"Vocabulario: {len(vetorizador.vocabulary_)} termos")
print()
print("Mapeamento de topicos:")
for topico, tema in MAPA_TOPICOS.items():
    print(f"  Topico {topico}: {tema}")


import unicodedata

def normalizar_texto(texto):
    texto = texto.replace('\xa0', ' ').replace('\n', ' ')
    texto = re.sub(r' +', ' ', texto).lower()
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )


def classificar_tema_lda(texto):
    """Retorna o tema regulatorio dominante do PADO via LDiA."""
    import numpy as np
    texto_norm = normalizar_texto(texto)
    vetor = vetorizador.transform([texto_norm])
    distribuicao = lda_modelo.transform(vetor)[0]
    topico_dominante = int(np.argmax(distribuicao))
    score = round(float(distribuicao[topico_dominante]), 3)
    tema = MAPA_TOPICOS.get(topico_dominante + 1, "Outro")
    return topico_dominante + 1, tema, score, distribuicao


print()
print("Funcao classificar_tema_lda pronta")

Modelo LDiA carregado
Topicos: 10
Vocabulario: 2000 termos

Mapeamento de topicos:
  Topico 1: Despacho de Instauracao
  Topico 2: Outro
  Topico 3: Outro
  Topico 4: Auto de Infracao
  Topico 5: Informe
  Topico 6: Oficio de Notificacao
  Topico 7: Certidao de Intimacao
  Topico 8: Outro
  Topico 9: Outro
  Topico 10: Outro

Funcao classificar_tema_lda pronta


In [39]:
# 5. CLASSIFICACAO TEMATICA DOS 104 PADOs VIA LDiA

import fitz

def extrair_texto_pado(pasta_pado, chars_min=100):
    """Extrai texto completo de todos os PDFs de uma pasta."""
    texto_total = ""
    for pdf in sorted(Path(pasta_pado).glob("*.pdf")):
        try:
            doc = fitz.open(str(pdf))
            for pagina in doc:
                t = pagina.get_text()
                if len(t.strip()) >= chars_min:
                    texto_total += t
            doc.close()
        except Exception:
            pass
    return texto_total.replace('\xa0', ' ')


pasta_raiz = Path(PASTA_PADOS)
subpastas = sorted([p for p in pasta_raiz.iterdir() if p.is_dir()])

print(f"Classificando temas de {len(subpastas)} PADOs...\n")

temas_pados = []
for pasta in subpastas:
    texto = extrair_texto_pado(pasta)
    if not texto.strip():
        temas_pados.append({
            "PADO": pasta.name,
            "Topico_LDA": 0,
            "Tema_LDA": "Sem texto extraivel",
            "Score_LDA": 0.0,
        })
        continue

    topico, tema, score, distribuicao = classificar_tema_lda(texto)
    temas_pados.append({
        "PADO": pasta.name,
        "Topico_LDA": topico,
        "Tema_LDA": tema,
        "Score_LDA": score,
    })
    print(f"{pasta.name[:45]:<45} | Topico {topico} | {tema} | score={score}")

df_temas = pd.DataFrame(temas_pados)
print(f"\n{len(df_temas)} PADOs classificados")
print("\nDistribuicao de temas:")
print(df_temas['Tema_LDA'].value_counts().to_string())

Classificando temas de 104 PADOs...

53000.062585_2010-06                          | Topico 8 | Outro | score=0.63
53500.007555_2026-55                          | Topico 6 | Oficio de Notificacao | score=0.767
53500.008890_2026-71                          | Topico 5 | Informe | score=0.501
53500.008991_2025-61                          | Topico 6 | Oficio de Notificacao | score=0.633
53500.008995_2025-49                          | Topico 6 | Oficio de Notificacao | score=0.613
53500.009858_2026-11                          | Topico 6 | Oficio de Notificacao | score=0.686
53500.011614_2022-66                          | Topico 5 | Informe | score=0.529
53500.014139_2023-61                          | Topico 1 | Despacho de Instauracao | score=1.0
53500.016622_2024-61                          | Topico 5 | Informe | score=0.884
53500.017655_2022-66                          | Topico 4 | Auto de Infracao | score=0.999
53500.019425_2026-65                          | Topico 5 | Informe | score=0.

In [40]:
# 6. RELATORIO XAI POR PADO
# Combina: tema LDiA + conformidade R1-R7 + fluxo processual

REQUISITOS_NOMES = {
    "R1_Documento_Motivador":  "R1 Documento Motivador",
    "R2_Analise_Previa":       "R2 Analise Previa",
    "R3_Despacho_Instauracao": "R3 Despacho Instauracao",
    "R4_Notificacao_Autuado":  "R4 Notificacao Autuado",
    "R5_Base_Legal":           "R5 Base Legal",
    "R6_Prazo_Defesa":         "R6 Prazo Defesa",
    "R7_Identificacao_Processo": "R7 Identificacao Processo",
}

def gerar_relatorio_pado(row_nb1, tema_row, atividades):
    """
    Gera relatorio XAI completo para um PADO.
    Combina conformidade R1-R7, tema LDiA e fluxo processual.
    """
    pado = row_nb1['PADO']
    conformidade = row_nb1.get('Conformidade', 'N/A')
    pct = row_nb1.get('Pct', 0)
    flag_ocr = row_nb1.get('flag_ocr', 'NAO')
    paginas_sem_texto = row_nb1.get('paginas_sem_texto', 0)

    # Tema regulatorio
    tema = tema_row['Tema_LDA'] if tema_row is not None else "Nao classificado"
    score_lda = tema_row['Score_LDA'] if tema_row is not None else 0.0
    topico = tema_row['Topico_LDA'] if tema_row is not None else 0

    # Fluxo processual
    tem_a1 = 'A1_Documento_Motivador' in atividades
    tem_a2 = 'A2_Analise_Previa' in atividades
    tem_a3 = 'A3_Despacho_Instauracao' in atividades
    tem_a4 = 'A4_Notificacao_Autuado' in atividades

    if tem_a3 and tem_a4:
        status_fluxo = "CONFORME"
    elif tem_a3 or tem_a4:
        status_fluxo = "PARCIAL"
    else:
        status_fluxo = "NAO CONFORME"

    # Requisitos nao conformes com explicacao
    requisitos_nao = []
    for req_id in REQUISITOS_NOMES:
        resultado = row_nb1.get(req_id, 'N/A')
        if resultado == 'NÃO':
            explicacao_col = f"{req_id}_explicacao"
            explicacao = row_nb1.get(explicacao_col, "Sem explicacao disponivel")
            requisitos_nao.append(f"{REQUISITOS_NOMES[req_id]}: {explicacao}")

    relatorio = {
        "PADO":                pado,
        "Conformidade_R1_R7":  conformidade,
        "Pct_Conformidade":    pct,
        "Status_Fluxo":        status_fluxo,
        "Tema_LDA":            tema,
        "Topico_LDA":          topico,
        "Score_LDA":           score_lda,
        "Flag_OCR":            flag_ocr,
        "Paginas_Sem_Texto":   paginas_sem_texto,
        "Atividades_Presentes": ", ".join(sorted(atividades)) if atividades else "Nenhuma",
        "A1_presente":         "SIM" if tem_a1 else "NAO",
        "A2_presente":         "SIM" if tem_a2 else "NAO",
        "A3_presente":         "SIM" if tem_a3 else "NAO",
        "A4_presente":         "SIM" if tem_a4 else "NAO",
        "Qtd_Requisitos_NAO":  len(requisitos_nao),
        "Explicacoes_NAO":     " | ".join(requisitos_nao) if requisitos_nao else "Todos os requisitos conformes",
    }

    return relatorio


# Gera relatorio para todos os PADOs
print("Gerando relatorios XAI...\n")
relatorios = []

for _, row_nb1 in df_nb1.iterrows():
    pado = row_nb1['PADO']

    # Busca tema LDiA
    tema_rows = df_temas[df_temas['PADO'] == pado]
    tema_row = tema_rows.iloc[0] if len(tema_rows) > 0 else None

    # Converte nome da pasta para formato case_id do event log
    case_id = pado.replace('_', '.', 1)
    atividades = atividades_por_pado.get(case_id, set())

    relatorio = gerar_relatorio_pado(row_nb1, tema_row, atividades)
    relatorios.append(relatorio)
    print(f"{pado[:45]:<45} | {relatorio['Conformidade_R1_R7']} | {relatorio['Status_Fluxo']} | {relatorio['Tema_LDA']}")
    
df_relatorio = pd.DataFrame(relatorios)
print(f"\n{len(df_relatorio)} relatorios gerados")

Gerando relatorios XAI...

53000.062585_2010-06                          | 4/7 | PARCIAL | Outro
53500.007555_2026-55                          | 6/7 | PARCIAL | Oficio de Notificacao
53500.008890_2026-71                          | 7/7 | CONFORME | Informe
53500.008991_2025-61                          | 6/7 | CONFORME | Oficio de Notificacao
53500.008995_2025-49                          | 6/7 | CONFORME | Oficio de Notificacao
53500.009858_2026-11                          | 7/7 | PARCIAL | Oficio de Notificacao
53500.011614_2022-66                          | 6/7 | CONFORME | Informe
53500.014139_2023-61                          | 7/7 | CONFORME | Despacho de Instauracao
53500.016622_2024-61                          | 7/7 | CONFORME | Informe
53500.017655_2022-66                          | 7/7 | CONFORME | Auto de Infracao
53500.019425_2026-65                          | 7/7 | PARCIAL | Informe
53500.021979_2025-41                          | 5/7 | CONFORME | Oficio de Notificacao
53500.02

In [41]:
# 7. SALVAR RELATORIO XAI EM EXCEL
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl import load_workbook

def limpar_para_excel(valor):
    if not isinstance(valor, str):
        return valor
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', valor)

# Limpa caracteres invalidos
df_relatorio_limpo = df_relatorio.copy()
for col in df_relatorio_limpo.columns:
    df_relatorio_limpo[col] = df_relatorio_limpo[col].apply(limpar_para_excel)

with pd.ExcelWriter(ARQUIVO_SAIDA, engine='openpyxl') as writer:
    df_relatorio_limpo.to_excel(writer, sheet_name='Relatorio XAI', index=False)

    ws = writer.sheets['Relatorio XAI']

    # Cabecalho
    azul    = PatternFill(start_color='2E5090', end_color='2E5090', fill_type='solid')
    verde   = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    vermelho = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    amarelo = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')

    for cell in ws[1]:
        cell.fill = azul
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    # Colore status de conformidade e fluxo
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if cell.value == 'CONFORME':
                cell.fill = verde
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'NAO CONFORME':
                cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'PARCIAL':
                cell.fill = amarelo
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'SIM':
                cell.fill = verde
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'NAO':
                cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')

    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(
            max(len(str(c.value or '')) for c in col) + 4, 50
        )

print(f"Relatorio XAI salvo em: {Path(ARQUIVO_SAIDA).resolve()}")
print(f"Total de PADOs no relatorio: {len(df_relatorio_limpo)}")
print()
print("Distribuicao Status Fluxo:")
print(df_relatorio_limpo['Status_Fluxo'].value_counts().to_string())
print()
print("Distribuicao Temas LDiA:")
print(df_relatorio_limpo['Tema_LDA'].value_counts().to_string())

Relatorio XAI salvo em: D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\relatorio_xai_pados.xlsx
Total de PADOs no relatorio: 104

Distribuicao Status Fluxo:
Status_Fluxo
CONFORME        65
PARCIAL         38
NAO CONFORME     1

Distribuicao Temas LDiA:
Tema_LDA
Outro                      61
Oficio de Notificacao      15
Despacho de Instauracao    14
Informe                    10
Certidao de Intimacao       3
Auto de Infracao            1


In [42]:
# 8. ESTATISTICAS GERAIS DO RELATORIO XAI

print('=' * 60)
print('RELATORIO XAI — ESTATISTICAS GERAIS')
print('=' * 60)
print(f'Total de PADOs analisados: {len(df_relatorio)}')
print()

# Conformidade R1-R7
print('CONFORMIDADE R1-R7 (Semantic Frames):')
print('-' * 60)
total = len(df_relatorio)
pct_100 = (df_relatorio['Pct_Conformidade'] == 100).sum()
pct_abaixo = (df_relatorio['Pct_Conformidade'] < 100).sum()
media_pct = df_relatorio['Pct_Conformidade'].mean()
print(f'  Media de conformidade    : {media_pct:.1f}%')
print(f'  PADOs 100% conformes     : {pct_100}/{total} ({pct_100/total*100:.1f}%)')
print(f'  PADOs abaixo de 100%     : {pct_abaixo}/{total} ({pct_abaixo/total*100:.1f}%)')
print()

# Status de fluxo
print('STATUS DE FLUXO PROCESSUAL (Conformance Checking):')
print('-' * 60)
for status, count in df_relatorio['Status_Fluxo'].value_counts().items():
    print(f'  {status:<20}: {count}/{total} ({count/total*100:.1f}%)')
print()

# OCR
print('QUALIDADE DOS PDFs:')
print('-' * 60)
ocr_sim = (df_relatorio['Flag_OCR'] == 'SIM').sum()
ocr_nao = (df_relatorio['Flag_OCR'] == 'NAO').sum()
print(f'  PDFs com paginas escaneadas : {ocr_sim}/{total} ({ocr_sim/total*100:.1f}%)')
print(f'  PDFs totalmente extraiveis  : {ocr_nao}/{total} ({ocr_nao/total*100:.1f}%)')
print()

# Temas LDiA
print('CLASSIFICACAO TEMATICA (LDiA):')
print('-' * 60)
for tema, count in df_relatorio['Tema_LDA'].value_counts().items():
    barra = '█' * int(count / total * 40)
    print(f'  {tema:<30} {barra} {count} ({count/total*100:.1f}%)')
print()

# PADOs com mais requisitos NAO conformes
print('PADOs COM MAIS REQUISITOS NAO CONFORMES:')
print('-' * 60)
df_relatorio_sorted = df_relatorio.sort_values('Qtd_Requisitos_NAO', ascending=False)
for _, row in df_relatorio_sorted[df_relatorio_sorted['Qtd_Requisitos_NAO'] > 0].head(10).iterrows():
    print(f'  {row["PADO"][:45]:<45} | {row["Qtd_Requisitos_NAO"]} NAO | {row["Conformidade_R1_R7"]}')

RELATORIO XAI — ESTATISTICAS GERAIS
Total de PADOs analisados: 104

CONFORMIDADE R1-R7 (Semantic Frames):
------------------------------------------------------------
  Media de conformidade    : 81.4%
  PADOs 100% conformes     : 27/104 (26.0%)
  PADOs abaixo de 100%     : 77/104 (74.0%)

STATUS DE FLUXO PROCESSUAL (Conformance Checking):
------------------------------------------------------------
  CONFORME            : 65/104 (62.5%)
  PARCIAL             : 38/104 (36.5%)
  NAO CONFORME        : 1/104 (1.0%)

QUALIDADE DOS PDFs:
------------------------------------------------------------
  PDFs com paginas escaneadas : 73/104 (70.2%)
  PDFs totalmente extraiveis  : 31/104 (29.8%)

CLASSIFICACAO TEMATICA (LDiA):
------------------------------------------------------------
  Outro                          ███████████████████████ 61 (58.7%)
  Oficio de Notificacao          █████ 15 (14.4%)
  Despacho de Instauracao        █████ 14 (13.5%)
  Informe                        ███ 10 (9.6%

In [43]:
# 10. SCORE COMPOSTO DAS 3 CAMADAS
# Combina conformidade R1-R7 (Camada 1), fluxo processual (Camada 2)
# e qualidade de dados (flag_ocr) em um score unico por PADO

def calcular_score_composto(row):
    """
    Score composto ponderado das 3 camadas.

    Camada 1 — Conformidade R1-R7 (peso 0.5)
      Fonte: Pct_Conformidade do notebook 1
      0 a 100% normalizado para 0 a 1

    Camada 2 — Fluxo processual (peso 0.35)
      Fonte: Status_Fluxo do notebook 2
      CONFORME = 1.0 | PARCIAL = 0.5 | NAO CONFORME = 0.0

    Camada 3 — Qualidade de dados (peso 0.15)
      Fonte: Flag_OCR do notebook 1
      NAO = 1.0 (sem paginas escaneadas)
      SIM = penalidade proporcional ao numero de paginas sem texto
    """
    # Camada 1
    score_c1 = float(row['Pct_Conformidade']) / 100.0

    # Camada 2
    status_fluxo = row['Status_Fluxo']
    if status_fluxo == 'CONFORME':
        score_c2 = 1.0
    elif status_fluxo == 'PARCIAL':
        score_c2 = 0.5
    else:
        score_c2 = 0.0

    # Camada 3 — penalidade por OCR
    flag_ocr = row['Flag_OCR']
    if flag_ocr == 'NAO':
        score_c3 = 1.0
    else:
        paginas = int(row['Paginas_Sem_Texto']) if row['Paginas_Sem_Texto'] else 0
        # Penalidade progressiva: 0 paginas = 1.0, 50+ paginas = 0.0
        score_c3 = max(0.0, 1.0 - (paginas / 50.0))

    # Score composto ponderado
    score = (score_c1 * 0.50) + (score_c2 * 0.35) + (score_c3 * 0.15)
    return round(score, 3)


def classificar_score(score):
    if score >= 0.85:
        return "ALTO"
    elif score >= 0.65:
        return "MEDIO"
    elif score >= 0.40:
        return "BAIXO"
    else:
        return "CRITICO"


# Calcula score composto para todos os PADOs
df_relatorio['Score_Composto'] = df_relatorio.apply(calcular_score_composto, axis=1)
df_relatorio['Nivel_Score'] = df_relatorio['Score_Composto'].apply(classificar_score)

# Estatisticas
print('=' * 60)
print('SCORE COMPOSTO — 3 CAMADAS')
print('=' * 60)
print(f'Media do score composto: {df_relatorio["Score_Composto"].mean():.3f}')
print(f'Score maximo           : {df_relatorio["Score_Composto"].max():.3f}')
print(f'Score minimo           : {df_relatorio["Score_Composto"].min():.3f}')
print()
print('Distribuicao por nivel:')
for nivel, count in df_relatorio['Nivel_Score'].value_counts().items():
    total = len(df_relatorio)
    barra = '█' * int(count / total * 40)
    print(f'  {nivel:<10} {barra} {count} ({count/total*100:.1f}%)')
print()
print('Top 10 PADOs com maior score:')
print('-' * 60)
top10 = df_relatorio.nlargest(10, 'Score_Composto')[
    ['PADO', 'Score_Composto', 'Nivel_Score', 'Conformidade_R1_R7', 'Status_Fluxo']
]
for _, row in top10.iterrows():
    print(f"  {row['PADO'][:40]:<40} | {row['Score_Composto']:.3f} | {row['Nivel_Score']} | {row['Conformidade_R1_R7']} | {row['Status_Fluxo']}")
print()
print('Top 10 PADOs com menor score:')
print('-' * 60)
bot10 = df_relatorio.nsmallest(10, 'Score_Composto')[
    ['PADO', 'Score_Composto', 'Nivel_Score', 'Conformidade_R1_R7', 'Status_Fluxo']
]
for _, row in bot10.iterrows():
    print(f"  {row['PADO'][:40]:<40} | {row['Score_Composto']:.3f} | {row['Nivel_Score']} | {row['Conformidade_R1_R7']} | {row['Status_Fluxo']}")

SCORE COMPOSTO — 3 CAMADAS
Media do score composto: 0.813
Score maximo           : 1.000
Score minimo           : 0.456

Distribuicao por nivel:
  MEDIO      █████████████████ 46 (44.2%)
  ALTO       ████████████████ 44 (42.3%)
  BAIXO      █████ 14 (13.5%)

Top 10 PADOs com maior score:
------------------------------------------------------------
  53500.008890_2026-71                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.014139_2023-61                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.016622_2024-61                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.034731_2025-41                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.036487_2023-99                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.099849_2023-52                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.102531_2024-47                     | 1.000 | ALTO | 7/7 | CONFORME
  53500.308521_2022-51                     | 1.000 | ALTO | 7/7 | CONFORME
  53504.001167_2026-21                     | 1.000

In [44]:
# 11. SALVAR EXCEL FINAL COM SCORE COMPOSTO E FORMATACAO COMPLETA
from openpyxl.styles import PatternFill, Font, Alignment

def limpar_para_excel(valor):
    if not isinstance(valor, str):
        return valor
    import re
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', valor)

df_relatorio_limpo2 = df_relatorio.copy()
for col in df_relatorio_limpo2.columns:
    df_relatorio_limpo2[col] = df_relatorio_limpo2[col].apply(limpar_para_excel)

with pd.ExcelWriter(ARQUIVO_SAIDA, engine='openpyxl') as writer:
    df_relatorio_limpo2.to_excel(writer, sheet_name='Relatorio XAI', index=False)

    ws = writer.sheets['Relatorio XAI']

    # Cores
    azul     = PatternFill(start_color='2E5090', end_color='2E5090', fill_type='solid')
    verde    = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    vermelho = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    amarelo  = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')
    laranja  = PatternFill(start_color='F4B942', end_color='F4B942', fill_type='solid')
    cinza    = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')

    # Cabecalho
    for cell in ws[1]:
        cell.fill = azul
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    # Conteudo
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            v = cell.value
            if v == 'CONFORME' or v == 'ALTO' or v == 'SIM':
                cell.fill = verde
                cell.alignment = Alignment(horizontal='center')
            elif v == 'NAO CONFORME' or v == 'CRITICO' or v == 'NAO':
                cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')
            elif v == 'PARCIAL' or v == 'MEDIO':
                cell.fill = amarelo
                cell.alignment = Alignment(horizontal='center')
            elif v == 'BAIXO':
                cell.fill = laranja
                cell.alignment = Alignment(horizontal='center')
            elif isinstance(v, float) and 0 < v <= 1.0:
                # Score composto — gradiente visual
                if v >= 0.85:
                    cell.fill = verde
                elif v >= 0.65:
                    cell.fill = amarelo
                elif v >= 0.40:
                    cell.fill = laranja
                else:
                    cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')

    # Largura das colunas
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(
            max(len(str(c.value or '')) for c in col) + 4, 50
        )

print(f"Excel final salvo com formatacao completa em: {ARQUIVO_SAIDA}")
print(f"Total de PADOs: {len(df_relatorio_limpo2)}")
print(f"Colunas: {len(df_relatorio_limpo2.columns)}")

Excel final salvo com formatacao completa em: relatorio_xai_pados.xlsx
Total de PADOs: 104
Colunas: 18


In [46]:
print("Colunas do df_relatorio:")
for col in df_relatorio.columns:
    print(f"  '{col}'")

Colunas do df_relatorio:
  'PADO'
  'Conformidade_R1_R7'
  'Pct_Conformidade'
  'Status_Fluxo'
  'Tema_LDA'
  'Topico_LDA'
  'Score_LDA'
  'Flag_OCR'
  'Paginas_Sem_Texto'
  'Atividades_Presentes'
  'A1_presente'
  'A2_presente'
  'A3_presente'
  'A4_presente'
  'Qtd_Requisitos_NAO'
  'Explicacoes_NAO'
  'Score_Composto'
  'Nivel_Score'


In [47]:
# Célula 12 — Relatório HTML com explicações humanizadas e gráficos Chart.js
# Cole esse código no notebook 3 como célula 12

ARQUIVO_HTML = "relatorio_xai_pados.html"

# ─── EXPLICAÇÕES HUMANIZADAS ────────────────────────────────────────────────
# Linguagem baseada nos normativos da ANATEL e terminologia dos agentes

EXPLICACOES_HUMANIZADAS = {
    "R1_Documento_Motivador": {
        "ocr": "O documento que motivou a abertura deste PADO (Relatório de Fiscalização, Auto de Infração ou Informe) está em formato escaneado e não pôde ser lido automaticamente. Recomenda-se verificação manual do documento físico para confirmar a presença do elemento motivador.",
        "etapa_ausente": "Não foi identificado na pasta do processo nenhum documento da etapa motivadora (Relatório de Fiscalização, Auto de Infração em campo ou Informe técnico). Conforme o Art. 173 da LGT, todo PADO deve ser precedido de elemento que demonstre indícios de descumprimento de obrigações.",
        "nao_localizado": "O documento motivador foi identificado na pasta, porém o conteúdo não apresenta de forma clara os indícios de irregularidade ou o fato que motivou a instauração do procedimento.",
    },
    "R2_Analise_Previa": {
        "ocr": "O Informe técnico com análise prévia está em formato escaneado e não pôde ser processado. A análise prévia é obrigatória nos PADOs Tipo A (por Informe), conforme o RASA Art. 15.",
        "etapa_ausente": "Não foi localizado Informe técnico com análise prévia na pasta do processo. Nos procedimentos Tipo A, o Informe deve conter a proposta de instauração fundamentada antes do Despacho Ordinatório.",
        "nao_localizado": "O Informe foi identificado, porém não contém de forma explícita a proposta de instauração do PADO ou a conclusão pela abertura do procedimento sancionador.",
    },
    "R3_Despacho_Instauracao": {
        "ocr": "O Despacho Ordinatório de Instauração está em formato escaneado e não foi possível verificar seu conteúdo. Esse ato formal é obrigatório para dar início ao procedimento administrativo sancionador.",
        "etapa_ausente": "Não foi localizado Despacho Ordinatório de Instauração na pasta do processo. Esse documento é o ato formal que instaura o PADO e deve preceder qualquer notificação ao autuado, conforme a Lei 9.784/99.",
        "nao_localizado": "O Despacho Ordinatório foi localizado, mas não apresenta de forma clara a determinação de instauração do Procedimento para Apuração de Descumprimento de Obrigações (PADO).",
    },
    "R4_Notificacao_Autuado": {
        "ocr": "O documento de notificação (Ofício ou Certidão de Intimação) está em formato escaneado. A notificação ao autuado é garantia constitucional do contraditório e ampla defesa (CF/88, Art. 5º, LV).",
        "etapa_ausente": "Não foi localizado documento de notificação ao autuado (Ofício de Notificação, Certidão de Intimação ou Aviso de Recebimento) na pasta do processo. A ausência de notificação configura violação ao princípio do contraditório.",
        "nao_localizado": "O documento de notificação foi localizado, porém não demonstra de forma clara a comunicação ao autuado sobre a instauração do procedimento e o prazo para apresentação de defesa.",
    },
    "R5_Base_Legal": {
        "ocr": "O documento com a fundamentação legal está em formato escaneado. A base legal das infrações imputadas deve ser citada expressamente no processo, conforme o Art. 50 da Lei 9.784/99.",
        "etapa_ausente": "Não foi identificada a citação expressa da base legal das infrações nos documentos da etapa esperada (Despacho ou Ofício de Notificação). A fundamentação normativa é requisito de validade do ato administrativo.",
        "nao_localizado": "Os documentos da etapa esperada foram localizados, porém não apresentam de forma clara os dispositivos normativos (artigos de lei, resoluções ou regulamentos) que fundamentam as infrações imputadas.",
    },
    "R6_Prazo_Defesa": {
        "ocr": "O Ofício de Notificação com a fixação do prazo de defesa está em formato escaneado. O prazo para apresentação de defesa é garantia processual prevista no RASA Art. 33.",
        "etapa_ausente": "Não foi localizado documento que estabeleça o prazo para apresentação de defesa pelo autuado. O prazo de 15 (quinze) dias para defesa é requisito obrigatório conforme o RASA Art. 33 e o Regimento Interno da ANATEL.",
        "nao_localizado": "O Ofício de Notificação foi localizado, porém não contém de forma explícita a fixação do prazo para o autuado oferecer defesa e produzir as provas que julgar cabíveis.",
    },
    "R7_Identificacao_Processo": {
        "ocr": "O documento com a identificação do processo está em formato escaneado. O número do processo SEI e os dados do autuado devem estar identificados em todos os documentos do PADO.",
        "etapa_ausente": "Não foi possível identificar o número do processo e os dados do autuado (razão social, CNPJ/CPF) nos documentos analisados. A identificação clara é exigência da Lei 9.784/99 para validade do processo administrativo.",
        "nao_localizado": "Os documentos foram localizados, porém não apresentam de forma clara o número do processo SEI e/ou os dados completos do autuado (razão social e CNPJ/CPF).",
    },
}

def humanizar_explicacao(req_id, explicacao_raw):
    """Converte explicação técnica em linguagem acessível para auditores."""
    if not explicacao_raw or explicacao_raw == "Todos os requisitos conformes":
        return "Requisito verificado e em conformidade com os normativos aplicáveis."

    exp = str(explicacao_raw)
    mapa = EXPLICACOES_HUMANIZADAS.get(req_id, {})

    if "PDF com" in exp and "escaneado" in exp:
        return mapa.get("ocr", exp)
    elif "Etapa processual ausente" in exp:
        return mapa.get("etapa_ausente", exp)
    elif "nao localizado" in exp or "não localizado" in exp:
        return mapa.get("nao_localizado", exp)
    return exp

# Carrega colunas de resultado por requisito do Excel do notebook 1
df_req = pd.read_excel(ARQUIVO_NB1, sheet_name='Detalhes e Evidências')[
    ['PADO', 'R1_Documento_Motivador', 'R2_Analise_Previa', 'R3_Despacho_Instauracao',
     'R4_Notificacao_Autuado', 'R5_Base_Legal', 'R6_Prazo_Defesa', 'R7_Identificacao_Processo']
]

# Mescla com df_relatorio
df_relatorio_completo = df_relatorio.merge(df_req, on='PADO', how='left')

def gerar_html_completo(df):
    total = len(df)
    media_score = df['Score_Composto'].mean()
    conformes = (df['Status_Fluxo'] == 'CONFORME').sum()
    parciais = (df['Status_Fluxo'] == 'PARCIAL').sum()
    nao_conformes = (df['Status_Fluxo'] == 'NAO CONFORME').sum()
    pct_100 = (df['Pct_Conformidade'] == 100).sum()
    alto = (df['Nivel_Score'] == 'ALTO').sum()
    medio = (df['Nivel_Score'] == 'MEDIO').sum()
    baixo = (df['Nivel_Score'] == 'BAIXO').sum()

    # Dados para graficos
    import json

    # Grafico 1 — Distribuicao de nivel de score
    niveis_labels = ['ALTO', 'MEDIO', 'BAIXO', 'CRITICO']
    niveis_data = [int(alto), int(medio), int(baixo), 0]

    # Grafico 2 — Conformidade por requisito
    req_ids = [
        'R1_Documento_Motivador', 'R2_Analise_Previa', 'R3_Despacho_Instauracao',
        'R4_Notificacao_Autuado', 'R5_Base_Legal', 'R6_Prazo_Defesa', 'R7_Identificacao_Processo'
    ]
    req_labels = ['R1 Doc Motivador', 'R2 Análise Prévia', 'R3 Despacho', 'R4 Notificação', 'R5 Base Legal', 'R6 Prazo Defesa', 'R7 Identificação']
    req_sim = [int((df[r] == 'SIM').sum()) for r in req_ids]
    req_nao = [int((df[r] == 'NÃO').sum()) for r in req_ids]

    # Grafico 3 — Status de fluxo
    fluxo_data = [int(conformes), int(parciais), int(nao_conformes)]

    # Grafico 4 — Temas LDiA
    temas_vc = df['Tema_LDA'].value_counts()
    temas_labels = list(temas_vc.index)
    temas_data = [int(v) for v in temas_vc.values]

    # Grafico 5 — Score composto histograma (faixas)
    faixas = ['0.40-0.55', '0.55-0.65', '0.65-0.75', '0.75-0.85', '0.85-1.00']
    faixas_data = [
        int(((df['Score_Composto'] >= 0.40) & (df['Score_Composto'] < 0.55)).sum()),
        int(((df['Score_Composto'] >= 0.55) & (df['Score_Composto'] < 0.65)).sum()),
        int(((df['Score_Composto'] >= 0.65) & (df['Score_Composto'] < 0.75)).sum()),
        int(((df['Score_Composto'] >= 0.75) & (df['Score_Composto'] < 0.85)).sum()),
        int(((df['Score_Composto'] >= 0.85) & (df['Score_Composto'] <= 1.00)).sum()),
    ]

    # Gera linhas da tabela com explicacoes humanizadas
    linhas = ""
    for _, row in df.iterrows():
        nivel = row['Nivel_Score']
        cor_nivel = {'ALTO': '#22c55e', 'MEDIO': '#f59e0b', 'BAIXO': '#ef4444', 'CRITICO': '#7f1d1d'}.get(nivel, '#6b7280')
        cor_fluxo = {'CONFORME': '#22c55e', 'PARCIAL': '#f59e0b', 'NAO CONFORME': '#ef4444'}.get(row['Status_Fluxo'], '#6b7280')
        ocr_badge = '<span class="badge badge-warn">OCR</span>' if row['Flag_OCR'] == 'SIM' else ''

        # Requisitos nao conformes com explicacao humanizada
        reqs_nao_html = ""
        for req_id in req_ids:
            resultado = row.get(req_id, 'N/A')
            if resultado == 'NÃO':
                col_exp = f"{req_id}_explicacao"
                exp_raw = row.get(col_exp, '')
                exp_human = humanizar_explicacao(req_id, exp_raw)
                nome_curto = req_id.replace('_', ' ').replace('R1 ', 'R1 ').split('_')[0]
                label = req_id.replace('_', ' ').title().replace('R1 ', 'R1: ').replace('R2 ', 'R2: ').replace('R3 ', 'R3: ').replace('R4 ', 'R4: ').replace('R5 ', 'R5: ').replace('R6 ', 'R6: ').replace('R7 ', 'R7: ')
                reqs_nao_html += f"""
                <div class="req-item">
                    <div class="req-label"><span class="req-badge">NAO</span> {req_id.replace('_', ' ')}</div>
                    <div class="req-exp">{exp_human}</div>
                </div>"""

        if not reqs_nao_html:
            reqs_nao_html = '<div class="req-ok">✓ Todos os requisitos normativos verificados e em conformidade.</div>'

        atividades_fmt = str(row['Atividades_Presentes']).replace('_', ' ').replace('A1 Documento Motivador', '✓ A1 Documento Motivador').replace('A2 Analise Previa', '✓ A2 Análise Prévia').replace('A3 Despacho Instauracao', '✓ A3 Despacho Instauração').replace('A4 Notificacao Autuado', '✓ A4 Notificação Autuado')

        linhas += f"""
        <tr class="main-row" onclick="toggleDetalhe('{row['PADO']}')">
            <td><span class="pado-id">{row['PADO']}</span>{ocr_badge}</td>
            <td>
                <div class="score-bar-wrap">
                    <div class="score-bar"><div class="score-fill" style="width:{row['Pct_Conformidade']}%;background:{cor_nivel}"></div></div>
                    <span class="score-val">{row['Conformidade_R1_R7']}</span>
                </div>
            </td>
            <td><span class="status-badge" style="background:{cor_fluxo}20;color:{cor_fluxo};border:1px solid {cor_fluxo}40">{row['Status_Fluxo']}</span></td>
            <td><span class="tema-tag">{row['Tema_LDA']}</span></td>
            <td><span class="score-num" style="color:{cor_nivel}">{row['Score_Composto']:.3f}</span></td>
            <td><span class="nivel-badge" style="background:{cor_nivel}20;color:{cor_nivel}">{nivel}</span></td>
        </tr>
        <tr class="detalhe-row" id="det-{row['PADO']}">
            <td colspan="6">
                <div class="detalhe-box">
                    <div class="detalhe-section">
                        <div class="detalhe-titulo">Fluxo Processual</div>
                        <div class="atividades">{atividades_fmt if row['Atividades_Presentes'] != 'Nenhuma' else '⚠ Nenhuma atividade identificada no event log'}</div>
                        <div style="margin-top:0.6rem;font-size:0.73rem;color:var(--text2)">
                            Páginas sem texto: <strong style="color:var(--text)">{row['Paginas_Sem_Texto']}</strong> &nbsp;|&nbsp;
                            Tema LDiA: <strong style="color:var(--accent2)">{row['Tema_LDA']}</strong> (score {row['Score_LDA']})
                        </div>
                    </div>
                    <div class="detalhe-section detalhe-reqs">
                        <div class="detalhe-titulo">Análise de Conformidade — Requisitos Não Atendidos</div>
                        {reqs_nao_html}
                    </div>
                </div>
            </td>
        </tr>"""

    html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Framework de Transparência em IA — PADOs ANATEL</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Mono:ital,wght@0,400;0,500;1,400&family=Sora:wght@300;400;500;600;700&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root {{
  --bg: #0b0d14;
  --surface: #13151f;
  --surface2: #1c1f2e;
  --surface3: #242840;
  --border: #2a2d42;
  --text: #e2e8f0;
  --text2: #7c85a8;
  --accent: #7c6af7;
  --accent2: #a89ff8;
  --green: #22c55e;
  --yellow: #f59e0b;
  --red: #ef4444;
  --blue: #38bdf8;
}}
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ font-family:'Sora',sans-serif; background:var(--bg); color:var(--text); min-height:100vh; }}

/* HEADER */
.header {{
  background: linear-gradient(135deg, #0f0c29 0%, #1a1d35 50%, #13151f 100%);
  border-bottom: 1px solid var(--border);
  padding: 2rem 3rem 1.5rem;
  position: relative;
  overflow: hidden;
}}
.header::before {{
  content: '';
  position: absolute;
  top: -60px; right: -60px;
  width: 280px; height: 280px;
  background: radial-gradient(circle, #7c6af720 0%, transparent 70%);
  pointer-events: none;
}}
.header-top {{ display: flex; justify-content: space-between; align-items: flex-start; }}
.header h1 {{ font-size: 1.5rem; font-weight: 700; letter-spacing: -0.03em; }}
.header h1 em {{ color: var(--accent2); font-style: normal; }}
.header-sub {{ margin-top: 0.4rem; color: var(--text2); font-size: 0.75rem; font-family: 'DM Mono', monospace; letter-spacing: 0.02em; }}
.tag-anatel {{ background: var(--accent)20; color: var(--accent2); border: 1px solid var(--accent)40; font-size: 0.65rem; padding: 0.2rem 0.7rem; border-radius: 20px; font-family: 'DM Mono', monospace; letter-spacing: 0.05em; font-weight: 500; }}

/* METRICAS */
.metricas {{ display: grid; grid-template-columns: repeat(6, 1fr); gap: 0.8rem; padding: 1.2rem 3rem; background: var(--surface); border-bottom: 1px solid var(--border); }}
.card {{ background: var(--surface2); border: 1px solid var(--border); border-radius: 10px; padding: 0.9rem 1rem; position: relative; overflow: hidden; }}
.card::after {{ content: ''; position: absolute; bottom: 0; left: 0; right: 0; height: 2px; }}
.card.green::after {{ background: var(--green); }}
.card.yellow::after {{ background: var(--yellow); }}
.card.red::after {{ background: var(--red); }}
.card.accent::after {{ background: var(--accent); }}
.card-label {{ font-size: 0.65rem; color: var(--text2); text-transform: uppercase; letter-spacing: 0.1em; font-family: 'DM Mono', monospace; }}
.card-value {{ font-size: 1.7rem; font-weight: 700; margin-top: 0.15rem; font-family: 'DM Mono', monospace; letter-spacing: -0.03em; }}
.card.green .card-value {{ color: var(--green); }}
.card.yellow .card-value {{ color: var(--yellow); }}
.card.red .card-value {{ color: var(--red); }}
.card.accent .card-value {{ color: var(--accent2); }}

/* GRAFICOS */
.graficos {{ display: grid; grid-template-columns: 1fr 2fr 1fr; gap: 1rem; padding: 1.2rem 3rem; border-bottom: 1px solid var(--border); background: var(--bg); }}
.graficos-row2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 1rem; padding: 0 3rem 1.2rem; border-bottom: 1px solid var(--border); background: var(--bg); }}
.grafico-card {{ background: var(--surface); border: 1px solid var(--border); border-radius: 12px; padding: 1.2rem; }}
.grafico-titulo {{ font-size: 0.7rem; color: var(--text2); text-transform: uppercase; letter-spacing: 0.08em; font-family: 'DM Mono', monospace; margin-bottom: 1rem; }}
.grafico-card canvas {{ max-height: 200px; }}

/* FILTROS */
.filtros {{ padding: 0.8rem 3rem; display: flex; gap: 0.8rem; align-items: center; border-bottom: 1px solid var(--border); background: var(--surface); flex-wrap: wrap; }}
.filtros-label {{ font-size: 0.65rem; color: var(--text2); font-family: 'DM Mono', monospace; text-transform: uppercase; letter-spacing: 0.08em; }}
.filtros select, .filtros input {{
  background: var(--surface2); border: 1px solid var(--border);
  color: var(--text); padding: 0.35rem 0.7rem; border-radius: 6px;
  font-size: 0.75rem; font-family: 'DM Mono', monospace; cursor: pointer;
}}
.filtros select:focus, .filtros input:focus {{ outline: none; border-color: var(--accent); }}
#contador {{ font-size: 0.72rem; color: var(--text2); margin-left: auto; font-family: 'DM Mono', monospace; }}

/* TABELA */
.tabela-wrap {{ padding: 1.2rem 3rem 2rem; }}
table {{ width: 100%; border-collapse: collapse; font-size: 0.8rem; }}
thead tr {{ background: var(--surface2); }}
th {{ padding: 0.7rem 1rem; text-align: left; color: var(--text2); font-size: 0.63rem; text-transform: uppercase; letter-spacing: 0.1em; font-family: 'DM Mono', monospace; font-weight: 500; border-bottom: 2px solid var(--border); white-space: nowrap; }}
tbody .main-row {{ cursor: pointer; transition: background 0.12s; border-bottom: 1px solid var(--border); }}
tbody .main-row:hover {{ background: var(--surface2); }}
td {{ padding: 0.65rem 1rem; vertical-align: middle; }}

.pado-id {{ font-family: 'DM Mono', monospace; font-size: 0.75rem; color: var(--accent2); }}
.badge {{ font-size: 0.65rem; padding: 0.15rem 0.5rem; border-radius: 20px; font-weight: 500; font-family: 'DM Mono', monospace; }}
.badge-warn {{ background: #f59e0b20; color: #f59e0b; border: 1px solid #f59e0b40; margin-left: 0.4rem; }}
.status-badge {{ font-size: 0.68rem; padding: 0.2rem 0.6rem; border-radius: 20px; font-weight: 600; font-family: 'DM Mono', monospace; white-space: nowrap; }}
.tema-tag {{ font-size: 0.7rem; color: var(--text2); font-family: 'DM Mono', monospace; }}
.score-bar-wrap {{ display: flex; align-items: center; gap: 0.5rem; }}
.score-bar {{ background: var(--border); border-radius: 4px; height: 4px; width: 70px; flex-shrink: 0; }}
.score-fill {{ height: 100%; border-radius: 4px; }}
.score-val {{ font-family: 'DM Mono', monospace; font-size: 0.75rem; color: var(--text2); white-space: nowrap; }}
.score-num {{ font-family: 'DM Mono', monospace; font-size: 0.82rem; font-weight: 600; }}
.nivel-badge {{ font-size: 0.65rem; padding: 0.2rem 0.6rem; border-radius: 20px; font-weight: 700; font-family: 'DM Mono', monospace; white-space: nowrap; }}

/* DETALHE */
.detalhe-row {{ display: none; }}
.detalhe-row.open {{ display: table-row; }}
.detalhe-box {{
  background: var(--surface2); border: 1px solid var(--border);
  border-radius: 10px; padding: 1.2rem 1.5rem; margin: 0.3rem 0 0.5rem;
  display: grid; grid-template-columns: 1fr 2fr; gap: 1.5rem;
}}
.detalhe-titulo {{ font-size: 0.65rem; color: var(--accent2); text-transform: uppercase; letter-spacing: 0.1em; font-family: 'DM Mono', monospace; margin-bottom: 0.7rem; font-weight: 600; }}
.detalhe-section {{ font-size: 0.78rem; color: var(--text2); line-height: 1.6; }}
.atividades {{ font-family: 'DM Mono', monospace; font-size: 0.72rem; color: var(--text); line-height: 1.8; }}

/* REQUISITOS */
.req-item {{ margin-bottom: 0.8rem; padding: 0.7rem 0.9rem; background: var(--surface3); border-radius: 8px; border-left: 3px solid var(--red); }}
.req-label {{ font-size: 0.68rem; font-family: 'DM Mono', monospace; color: var(--text); margin-bottom: 0.35rem; font-weight: 600; }}
.req-badge {{ background: #ef444420; color: var(--red); border: 1px solid #ef444440; font-size: 0.6rem; padding: 0.1rem 0.4rem; border-radius: 4px; margin-right: 0.4rem; }}
.req-exp {{ font-size: 0.74rem; color: var(--text2); line-height: 1.65; }}
.req-ok {{ padding: 0.7rem 0.9rem; background: #22c55e10; border-radius: 8px; border-left: 3px solid var(--green); font-size: 0.75rem; color: var(--green); font-family: 'DM Mono', monospace; }}

/* FOOTER */
.footer {{
  padding: 1rem 3rem; border-top: 1px solid var(--border);
  color: var(--text2); font-size: 0.68rem; font-family: 'DM Mono', monospace;
  display: flex; justify-content: space-between; background: var(--surface);
}}
</style>
</head>
<body>

<div class="header">
  <div class="header-top">
    <div>
      <h1>Framework de <em>Transparência em IA</em> — PADOs ANATEL</h1>
      <div class="header-sub">Mestrado PPGTI/IFPB · Fase de Instauração · {total} processos analisados · Adaptado do DERECHA (Amaral Cejas et al., 2023)</div>
    </div>
    <span class="tag-anatel">ANATEL · PADO · INSTAURAÇÃO</span>
  </div>
</div>

<div class="metricas">
  <div class="card accent"><div class="card-label">PADOs analisados</div><div class="card-value">{total}</div></div>
  <div class="card accent"><div class="card-label">Score médio</div><div class="card-value">{media_score:.3f}</div></div>
  <div class="card green"><div class="card-label">Fluxo conforme</div><div class="card-value">{conformes}</div></div>
  <div class="card yellow"><div class="card-label">Fluxo parcial</div><div class="card-value">{parciais}</div></div>
  <div class="card green"><div class="card-label">R1-R7 100%</div><div class="card-value">{pct_100}</div></div>
  <div class="card red"><div class="card-label">Nível BAIXO</div><div class="card-value">{baixo}</div></div>
</div>

<!-- GRAFICOS LINHA 1 -->
<div class="graficos">
  <div class="grafico-card">
    <div class="grafico-titulo">Nível de Conformidade</div>
    <canvas id="grafico-nivel"></canvas>
  </div>
  <div class="grafico-card">
    <div class="grafico-titulo">Conformidade por Requisito (R1–R7)</div>
    <canvas id="grafico-requisitos"></canvas>
  </div>
  <div class="grafico-card">
    <div class="grafico-titulo">Status do Fluxo Processual</div>
    <canvas id="grafico-fluxo"></canvas>
  </div>
</div>

<!-- GRAFICOS LINHA 2 -->
<div class="graficos-row2">
  <div class="grafico-card">
    <div class="grafico-titulo">Distribuição do Score Composto</div>
    <canvas id="grafico-score"></canvas>
  </div>
  <div class="grafico-card">
    <div class="grafico-titulo">Classificação Temática LDiA</div>
    <canvas id="grafico-temas"></canvas>
  </div>
</div>

<div class="filtros">
  <span class="filtros-label">Filtrar:</span>
  <select id="filtroNivel" onchange="filtrar()">
    <option value="">Todos os níveis</option>
    <option value="ALTO">ALTO</option>
    <option value="MEDIO">MEDIO</option>
    <option value="BAIXO">BAIXO</option>
  </select>
  <select id="filtroFluxo" onchange="filtrar()">
    <option value="">Todos os fluxos</option>
    <option value="CONFORME">CONFORME</option>
    <option value="PARCIAL">PARCIAL</option>
    <option value="NAO CONFORME">NAO CONFORME</option>
  </select>
  <select id="filtroTema" onchange="filtrar()">
    <option value="">Todos os temas LDiA</option>
    <option value="Informe">Informe</option>
    <option value="Despacho de Instauracao">Despacho</option>
    <option value="Oficio de Notificacao">Ofício</option>
    <option value="Auto de Infracao">Auto de Infração</option>
    <option value="Outro">Outro</option>
  </select>
  <input type="text" id="busca" placeholder="Buscar PADO..." oninput="filtrar()">
  <span id="contador">{total} processos</span>
</div>

<div class="tabela-wrap">
  <table>
    <thead>
      <tr>
        <th>Processo PADO</th>
        <th>Conformidade R1–R7</th>
        <th>Fluxo Processual</th>
        <th>Tema Regulatório</th>
        <th>Score Composto</th>
        <th>Nível</th>
      </tr>
    </thead>
    <tbody id="tbody">
      {linhas}
    </tbody>
  </table>
</div>

<div class="footer">
  <span>Framework de Verificação de Conformidade — Camadas 1 (SF), 2 (Conformance Checking) e 3 (XAI/LDiA)</span>
  <span>PPGTI/IFPB · Gerado automaticamente pelo notebook 3</span>
</div>

<script>
// ─── GRAFICOS ─────────────────────────────────────────────────────────────
const C = {{
  green: '#22c55e', yellow: '#f59e0b', red: '#ef4444',
  accent: '#7c6af7', accent2: '#a89ff8', blue: '#38bdf8',
  text2: '#7c85a8', border: '#2a2d42', bg2: '#1c1f2e'
}};

Chart.defaults.color = C.text2;
Chart.defaults.borderColor = C.border;
Chart.defaults.font.family = "'DM Mono', monospace";
Chart.defaults.font.size = 11;

// Grafico 1 — Donut nivel
new Chart(document.getElementById('grafico-nivel'), {{
  type: 'doughnut',
  data: {{
    labels: {json.dumps(niveis_labels)},
    datasets: [{{ data: {json.dumps(niveis_data)}, backgroundColor: [C.green, C.yellow, C.red, '#7f1d1d'], borderWidth: 0, hoverOffset: 6 }}]
  }},
  options: {{ plugins: {{ legend: {{ position: 'bottom', labels: {{ boxWidth: 10, padding: 12 }} }} }}, cutout: '65%' }}
}});

// Grafico 2 — Barras empilhadas requisitos
new Chart(document.getElementById('grafico-requisitos'), {{
  type: 'bar',
  data: {{
    labels: {json.dumps(req_labels)},
    datasets: [
      {{ label: 'Conforme', data: {json.dumps(req_sim)}, backgroundColor: C.green + 'cc', borderRadius: 4 }},
      {{ label: 'Não Conforme', data: {json.dumps(req_nao)}, backgroundColor: C.red + 'cc', borderRadius: 4 }}
    ]
  }},
  options: {{
    indexAxis: 'y', plugins: {{ legend: {{ position: 'bottom', labels: {{ boxWidth: 10 }} }} }},
    scales: {{ x: {{ stacked: true, grid: {{ color: C.border }} }}, y: {{ stacked: true, grid: {{ display: false }} }} }}
  }}
}});

// Grafico 3 — Donut fluxo
new Chart(document.getElementById('grafico-fluxo'), {{
  type: 'doughnut',
  data: {{
    labels: ['Conforme', 'Parcial', 'Não Conforme'],
    datasets: [{{ data: {json.dumps(fluxo_data)}, backgroundColor: [C.green, C.yellow, C.red], borderWidth: 0, hoverOffset: 6 }}]
  }},
  options: {{ plugins: {{ legend: {{ position: 'bottom', labels: {{ boxWidth: 10, padding: 12 }} }} }}, cutout: '65%' }}
}});

// Grafico 4 — Histograma score
new Chart(document.getElementById('grafico-score'), {{
  type: 'bar',
  data: {{
    labels: {json.dumps(faixas)},
    datasets: [{{ label: 'PADOs', data: {json.dumps(faixas_data)}, backgroundColor: [C.red + 'cc', C.yellow + 'cc', C.yellow + 'cc', C.green + '99', C.green + 'cc'], borderRadius: 6 }}]
  }},
  options: {{
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ grid: {{ display: false }} }}, y: {{ grid: {{ color: C.border }}, ticks: {{ stepSize: 5 }} }} }}
  }}
}});

// Grafico 5 — Barras temas LDiA
new Chart(document.getElementById('grafico-temas'), {{
  type: 'bar',
  data: {{
    labels: {json.dumps(temas_labels)},
    datasets: [{{ label: 'PADOs', data: {json.dumps(temas_data)}, backgroundColor: C.accent + 'cc', borderRadius: 6 }}]
  }},
  options: {{
    indexAxis: 'y',
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ grid: {{ color: C.border }} }}, y: {{ grid: {{ display: false }} }} }}
  }}
}});

// ─── TABELA ───────────────────────────────────────────────────────────────
function toggleDetalhe(pado) {{
  const det = document.getElementById('det-' + pado);
  det.classList.toggle('open');
}}

function filtrar() {{
  const nivel = document.getElementById('filtroNivel').value;
  const fluxo = document.getElementById('filtroFluxo').value;
  const tema = document.getElementById('filtroTema').value;
  const busca = document.getElementById('busca').value.toLowerCase();
  const rows = document.querySelectorAll('tbody .main-row');
  let visiveis = 0;
  rows.forEach(row => {{
    const txt = row.textContent;
    const ok = (!nivel || txt.includes(nivel)) &&
               (!fluxo || txt.includes(fluxo)) &&
               (!tema || txt.includes(tema)) &&
               (!busca || txt.toLowerCase().includes(busca));
    row.style.display = ok ? '' : 'none';
    const det = row.nextElementSibling;
    if (det) {{ if (!ok) det.classList.remove('open'); det.style.display = ok ? '' : 'none'; }}
    if (ok) visiveis++;
  }});
  document.getElementById('contador').textContent = visiveis + ' processos';
}}
</script>
</body>
</html>"""
    return html


# Gera e salva o HTML
html_content = gerar_html_completo(df_relatorio_completo)
with open(ARQUIVO_HTML, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Relatorio HTML salvo em: {ARQUIVO_HTML}")
print("Abra o arquivo no Chrome ou Edge para visualizar o dashboard completo")
print(f"Total de PADOs: {len(df_relatorio)}")

Relatorio HTML salvo em: relatorio_xai_pados.html
Abra o arquivo no Chrome ou Edge para visualizar o dashboard completo
Total de PADOs: 104


In [49]:
# ÚLTIMA CÉLULA — RELATÓRIO PDF POR PADO (CORRIGIDA)

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable)
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from datetime import datetime
import pandas as pd

# ── CONFIGURAÇÃO ──────────────────────────────────────────────────────────────
PADO_ALVO         = "53500.014139_2023-61"
ARQUIVO_SAIDA     = "resultado_requisitos_pados_sf.xlsx"
ARQUIVO_XAI       = "relatorio_xai_pados.xlsx"
ARQUIVO_PDF_SAIDA = f"relatorio_{PADO_ALVO}.pdf"

# ── CORES ─────────────────────────────────────────────────────────────────────
COR_PRIMARIA   = colors.HexColor('#1e3a5f')
COR_SECUNDARIA = colors.HexColor('#2e5090')
COR_VERDE      = colors.HexColor('#22c55e')
COR_VERMELHO   = colors.HexColor('#ef4444')
COR_AMARELO    = colors.HexColor('#f59e0b')
COR_CINZA      = colors.HexColor('#f1f5f9')
COR_CINZA2     = colors.HexColor('#e2e8f0')
COR_TEXTO      = colors.HexColor('#1e293b')
COR_TEXTO2     = colors.HexColor('#64748b')

# ── ESTILOS ───────────────────────────────────────────────────────────────────
styles = getSampleStyleSheet()

s_titulo = ParagraphStyle('titulo',
    fontName='Helvetica-Bold', fontSize=16, textColor=COR_PRIMARIA,
    spaceAfter=4, leading=20)

s_subtitulo = ParagraphStyle('subtitulo',
    fontName='Helvetica-Bold', fontSize=11, textColor=COR_SECUNDARIA,
    spaceAfter=6, spaceBefore=12, leading=14)

s_normal = ParagraphStyle('normal',
    fontName='Helvetica', fontSize=9, textColor=COR_TEXTO,
    spaceAfter=4, leading=13)

s_negrito = ParagraphStyle('negrito',
    fontName='Helvetica-Bold', fontSize=9, textColor=COR_TEXTO,
    spaceAfter=4, leading=13)

s_pequeno = ParagraphStyle('pequeno',
    fontName='Helvetica', fontSize=8, textColor=COR_TEXTO2,
    spaceAfter=3, leading=11)

s_center = ParagraphStyle('center',
    fontName='Helvetica', fontSize=9, textColor=COR_TEXTO,
    alignment=TA_CENTER, leading=13)

s_header = ParagraphStyle('header',
    fontName='Helvetica-Bold', fontSize=8, textColor=colors.white,
    alignment=TA_CENTER, leading=11)

s_explicacao = ParagraphStyle('explicacao',
    fontName='Helvetica', fontSize=8, textColor=COR_TEXTO,
    spaceAfter=3, leading=12, leftIndent=8)


# ── FUNÇÃO PRINCIPAL ──────────────────────────────────────────────────────────
def gerar_relatorio_pdf(pado_id, arquivo_saida):

    # Carrega dados do notebook 1
    df_matriz  = pd.read_excel(ARQUIVO_SAIDA, sheet_name='Matriz de Conformidade')
    df_detalhe = pd.read_excel(ARQUIVO_SAIDA, sheet_name='Detalhes e Evidências')

    # Carrega Status_Fluxo do relatório XAI e faz merge
    df_xai = pd.read_excel(ARQUIVO_XAI)[['PADO', 'Status_Fluxo']]
    df_detalhe = df_detalhe.merge(df_xai, on='PADO', how='left')

    row_mat = df_matriz[df_matriz['PADO'] == pado_id]
    row_det = df_detalhe[df_detalhe['PADO'] == pado_id]

    if row_mat.empty or row_det.empty:
        print(f"PADO '{pado_id}' não encontrado.")
        return

    r = row_mat.iloc[0]
    d = row_det.iloc[0]

    # Documento PDF
    doc = SimpleDocTemplate(
        arquivo_saida,
        pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
        title=f"Relátorio PADO {pado_id}",
        author="Framework de Controle Transparência em IA — EIXO 6/ANATEL",
    )

    story = []
    W = A4[0] - 4*cm

    # ── CABEÇALHO ─────────────────────────────────────────────────────────────
    cab_data = [[
        Paragraph('<b>ANATEL</b><br/><font size="7" color="#64748b">Agencia Nacional de Telecomunicações</font>', s_normal),
        Paragraph('<b>Framework de Controle e Transparência em IA</b><br/><font size="7" color="#64748b">Verificação de Conformidade em PADOs</font>', s_center),
        Paragraph(f'<font size="7" color="#64748b">Gerado em:<br/>{datetime.now().strftime("%d/%m/%Y %H:%M")}</font>',
            ParagraphStyle('right', fontName='Helvetica', fontSize=8, textColor=COR_TEXTO2, alignment=TA_RIGHT, leading=11)),
    ]]
    cab_table = Table(cab_data, colWidths=[W*0.3, W*0.4, W*0.3])
    cab_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), COR_CINZA),
        ('BOX', (0,0), (-1,-1), 0.5, COR_CINZA2),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING', (0,0), (-1,-1), 10),
    ]))
    story.append(cab_table)
    story.append(Spacer(1, 12))

    # ── TÍTULO ────────────────────────────────────────────────────────────────
    story.append(Paragraph('Relátorio de Conformidade — PADO', s_titulo))
    story.append(Paragraph(f'{pado_id}', ParagraphStyle('pado_id',
        fontName='Helvetica-Bold', fontSize=13, textColor=COR_SECUNDARIA,
        spaceAfter=4, leading=16)))
    story.append(HRFlowable(width=W, thickness=2, color=COR_PRIMARIA, spaceAfter=10))

    # ── SEÇÃO 1: IDENTIFICAÇÃO DO PROCESSO ────────────────────────────────────
    story.append(Paragraph('1. Identificação do Processo', s_subtitulo))

    info_data = [
        ['Processo SEI',           str(d.get('Processo', 'N/A'))],
        ['Autuado',                str(d.get('Autuado', 'N/A'))],
        ['Tipo de PADO',           str(d.get('Tipo_PADO', 'N/A'))],
        ['Qtd. de PDFs',           str(d.get('Qtd_PDFs', 'N/A'))],
        ['Tipos de Documentos',    str(d.get('Tipos_Documentos', 'N/A'))],
        ['Páginas sem texto (OCR)', f"{d.get('paginas_sem_texto', 0)} paginas — Flag OCR: {d.get('flag_ocr', 'NAO')}"],
    ]
    info_table = Table(info_data, colWidths=[W*0.28, W*0.72])
    info_table.setStyle(TableStyle([
        ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTNAME', (1,0), (1,-1), 'Helvetica'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('TEXTCOLOR', (0,0), (0,-1), COR_PRIMARIA),
        ('TEXTCOLOR', (1,0), (1,-1), COR_TEXTO),
        ('ROWBACKGROUNDS', (0,0), (-1,-1), [colors.white, COR_CINZA]),
        ('GRID', (0,0), (-1,-1), 0.3, COR_CINZA2),
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('LEFTPADDING', (0,0), (-1,-1), 8),
    ]))
    story.append(info_table)
    story.append(Spacer(1, 10))

    # ── SEÇÃO 2: RESUMO DE CONFORMIDADE ───────────────────────────────────────
    story.append(Paragraph('2. Resumo de Conformidade', s_subtitulo))

    status_fluxo = str(d.get('Status_Fluxo', 'N/A'))
    conformidade = str(r.get('Conformidade', 'N/A'))
    pct          = str(r.get('Pct', 'N/A'))
    cor_fluxo    = COR_VERDE if status_fluxo == 'CONFORME' else \
                   COR_AMARELO if status_fluxo == 'PARCIAL' else COR_VERMELHO

    resumo_data = [
        [Paragraph('<b>Conformidade R1-R7</b>', s_header),
         Paragraph('<b>% Conformidade</b>', s_header),
         Paragraph('<b>Status Fluxo Processual</b>', s_header)],
        [Paragraph(f'<b>{conformidade}</b>', ParagraphStyle('conf',
             fontName='Helvetica-Bold', fontSize=13, textColor=COR_PRIMARIA,
             alignment=TA_CENTER, leading=18)),
         Paragraph(f'<b>{pct}%</b>', ParagraphStyle('pct',
             fontName='Helvetica-Bold', fontSize=13, textColor=COR_PRIMARIA,
             alignment=TA_CENTER, leading=18)),
         Paragraph(f'<b>{status_fluxo}</b>', ParagraphStyle('sf',
             fontName='Helvetica-Bold', fontSize=11, textColor=cor_fluxo,
             alignment=TA_CENTER, leading=14))],
    ]
    resumo_table = Table(resumo_data, colWidths=[W*0.33, W*0.33, W*0.34])
    resumo_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), COR_PRIMARIA),
        ('BACKGROUND', (0,1), (-1,1), COR_CINZA),
        ('GRID', (0,0), (-1,-1), 0.3, COR_CINZA2),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    ]))
    story.append(resumo_table)
    story.append(Spacer(1, 10))

    # ── SEÇÃO 3: CONFORMIDADE R1-R7 ───────────────────────────────────────────
    story.append(Paragraph('3. Verificação de Conformidade — Requisitos R1 a R7', s_subtitulo))
    story.append(Paragraph(
        'Verificação realizada via Semantic Frames com sentence-transformers '
        '(paraphrase-multilingual-MiniLM-L12-v2). Threshold de similaridade de cosseno: 0.55. '
        'Adaptado do framework DERECHA (Amaral Cejas et al., 2023).',
        s_pequeno))
    story.append(Spacer(1, 6))

    REQUISITOS_INFO = {
        'R1_Documento_Motivador':    ('R1', 'Documento Motivador',    'LGT Art. 173'),
        'R2_Analise_Previa':         ('R2', 'Analise Previa',         'RASA Art. 15'),
        'R3_Despacho_Instauracao':   ('R3', 'Despacho de Instauracao','Lei 9.784/99'),
        'R4_Notificacao_Autuado':    ('R4', 'Notificacao ao Autuado', 'CF/88 Art. 5 LV'),
        'R5_Base_Legal':             ('R5', 'Base Legal Citada',      'Lei 9.784/99 Art. 50'),
        'R6_Prazo_Defesa':           ('R6', 'Prazo para Defesa',      'RASA Art. 33'),
        'R7_Identificacao_Processo': ('R7', 'Identificacao do Processo','Lei 9.784/99'),
    }

    req_rows = [[
        Paragraph('<b>Req.</b>', s_header),
        Paragraph('<b>Requisito</b>', s_header),
        Paragraph('<b>Base Legal</b>', s_header),
        Paragraph('<b>Resultado</b>', s_header),
        Paragraph('<b>Etapa Verificada</b>', s_header),
    ]]

    for req_id, (codigo, nome, base) in REQUISITOS_INFO.items():
        resultado = str(d.get(req_id, 'N/A'))
        etapa     = str(d.get(f'{req_id}_etapa', 'N/A'))
        cor_res   = COR_VERDE if resultado == 'SIM' else COR_VERMELHO

        req_rows.append([
            Paragraph(f'<b>{codigo}</b>', ParagraphStyle('cod',
                fontName='Helvetica-Bold', fontSize=9, textColor=COR_PRIMARIA,
                alignment=TA_CENTER, leading=13)),
            Paragraph(nome, s_normal),
            Paragraph(base, s_pequeno),
            Paragraph(f'<b>{resultado}</b>', ParagraphStyle('res',
                fontName='Helvetica-Bold', fontSize=10, textColor=cor_res,
                alignment=TA_CENTER, leading=13)),
            Paragraph(etapa[:60] if etapa else 'N/A', s_pequeno),
        ])

    req_table = Table(req_rows, colWidths=[W*0.07, W*0.28, W*0.18, W*0.12, W*0.35])
    req_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), COR_PRIMARIA),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, COR_CINZA]),
        ('GRID', (0,0), (-1,-1), 0.3, COR_CINZA2),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
        ('ALIGN', (0,0), (0,-1), 'CENTER'),
        ('ALIGN', (3,0), (3,-1), 'CENTER'),
    ]))
    story.append(req_table)
    story.append(Spacer(1, 10))

    # ── SEÇÃO 4: ANÁLISE DOS NÃO CONFORMES ────────────────────────────────────
    nao_conformes = [(req_id, info) for req_id, info in REQUISITOS_INFO.items()
                     if str(d.get(req_id, '')) == 'NÃO']

    if nao_conformes:
        story.append(Paragraph('4. Analise Detalhada dos Requisitos Não Conformes', s_subtitulo))
        story.append(Paragraph(
            'Para cada requisito não conforme, o framework identifica a causa raiz '
            'com base na etapa processual esperada e na presença de documentos na pasta.',
            s_pequeno))
        story.append(Spacer(1, 6))

        for req_id, (codigo, nome, base) in nao_conformes:
            explicacao = str(d.get(f'{req_id}_explicacao', 'Sem explicação disponível'))

            exp_data = [[
                Paragraph(f'<b>{codigo} — {nome}</b> | Base legal: {base}',
                    ParagraphStyle('exp_titulo', fontName='Helvetica-Bold', fontSize=9,
                        textColor=colors.white, leading=13)),
            ],[
                Paragraph(explicacao, s_explicacao),
            ]]
            exp_table = Table(exp_data, colWidths=[W])
            exp_table.setStyle(TableStyle([
                ('BACKGROUND', (0,0), (-1,0), COR_VERMELHO),
                ('BACKGROUND', (0,1), (-1,1), colors.HexColor('#fef2f2')),
                ('BOX', (0,0), (-1,-1), 0.5, COR_VERMELHO),
                ('LEFTPADDING', (0,0), (-1,-1), 10),
                ('RIGHTPADDING', (0,0), (-1,-1), 10),
                ('TOPPADDING', (0,0), (-1,0), 6),
                ('BOTTOMPADDING', (0,0), (-1,0), 6),
                ('TOPPADDING', (0,1), (-1,1), 8),
                ('BOTTOMPADDING', (0,1), (-1,1), 8),
            ]))
            story.append(exp_table)
            story.append(Spacer(1, 6))

    # ── SEÇÃO 5: FLUXO PROCESSUAL ─────────────────────────────────────────────
    secao_num = 5 if nao_conformes else 4
    story.append(Paragraph(f'{secao_num}. Conformance Checking — Fluxo Processual', s_subtitulo))
    story.append(Paragraph(
        'Verificação do fluxo processual via Process Mining (pm4py) com modelo normativo manual. '
        'Sequência esperada: Tipo A (A2-A3-A4) | Tipo B (A1-A3-A4). '
        'Baseado em van der Aalst (2016) — Process Mining: Data Science in Action.',
        s_pequeno))
    story.append(Spacer(1, 6))

    cor_fluxo2 = COR_VERDE if status_fluxo == 'CONFORME' else \
                 COR_AMARELO if status_fluxo == 'PARCIAL' else COR_VERMELHO

    fluxo_data = [
        [Paragraph('<b>Tipo de PADO</b>', s_header),
         Paragraph('<b>Status do Fluxo Processual</b>', s_header)],
        [Paragraph(str(d.get('Tipo_PADO', 'N/A')), s_center),
         Paragraph(f'<b>{status_fluxo}</b>', ParagraphStyle('sf2',
             fontName='Helvetica-Bold', fontSize=11, textColor=cor_fluxo2,
             alignment=TA_CENTER, leading=14))],
    ]
    fluxo_table = Table(fluxo_data, colWidths=[W*0.4, W*0.6])
    fluxo_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), COR_PRIMARIA),
        ('BACKGROUND', (0,1), (-1,1), COR_CINZA),
        ('GRID', (0,0), (-1,-1), 0.3, COR_CINZA2),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 10),
        ('BOTTOMPADDING', (0,0), (-1,-1), 10),
    ]))
    story.append(fluxo_table)
    story.append(Spacer(1, 10))

    # ── RODAPÉ ────────────────────────────────────────────────────────────────
    story.append(HRFlowable(width=W, thickness=0.5, color=COR_CINZA2, spaceAfter=6))
    story.append(Paragraph(
        f'Relátorio gerado automaticamente pelo Framework de Controle e Transparência em IA. '
        f'Adaptado do DERECHA (Amaral Cejas et al., 2023, IEEE TSE vol. 49 no. 9). '
        f'Gerado em {datetime.now().strftime("%d/%m/%Y as %H:%M")}.',
        s_pequeno))

    doc.build(story)
    print(f'Relátorio PDF gerado: {arquivo_saida}')
    print(f'PADO: {pado_id}')
    print(f'Conformidade R1-R7: {r.get("Conformidade")} ({pct}%)')
    print(f'Status fluxo: {status_fluxo}')


# ── EXECUTA ───────────────────────────────────────────────────────────────────
gerar_relatorio_pdf(PADO_ALVO, ARQUIVO_PDF_SAIDA)

Relátorio PDF gerado: relatorio_53500.014139_2023-61.pdf
PADO: 53500.014139_2023-61
Conformidade R1-R7: 7/7 (100%)
Status fluxo: CONFORME
